In [ ]:
import os
#os.environ["TF_GPU_THREAD_MODE"] = "gpu_private"


import numpy as np
import tensorflow as tf
from tensorflow.python.client import device_lib
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


gpu_options = tf.compat.v1.GPUOptions(per_process_gpu_memory_fraction=0.9)
sess = tf.compat.v1.Session(config=tf.compat.v1.ConfigProto(gpu_options=gpu_options))

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.test.is_gpu_available())

# Get detailed device information
print(device_lib.list_local_devices())



set_size=400

shape = (set_size, 128, 128, 98)
dataset = np.random.uniform(low=-1, high=1, size=shape).astype(np.float16)
shape = (set_size)
truth_temp = np.random.uniform(low=0, high=48, size=shape).astype(int)

truth = np.zeros((len(truth_temp), 49), dtype=int)
truth[np.arange(len(truth_temp)), truth_temp] = 1
# Model creation
def create_model(input_shape=(128, 128, 98), num_classes=49):
    base_model = MobileNetV2(input_shape=input_shape, include_top=False, weights=None)
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    logits = Dense(num_classes, activation=None)(x)  # Output 49 logits
    model = Model(inputs=base_model.input, outputs=logits)
    return model

# Loss function for ordinal regression
#@tf.keras.saving.register_keras_serializable(package="Custom")
def ordinal_loss(y_true, y_pred):
    l2_loss = tf.reduce_mean(tf.square(y_pred - y_true), axis=None)
    return l2_loss


# Create model
model = create_model()

# Compile the model
model.compile(
    optimizer=Adam(learning_rate=1e-3, beta_1=0.5, beta_2=0.999),
    loss=ordinal_loss
)

# Training
batch_size = 20
global_steps = 100
steps_per_epoch = set_size // batch_size  # Example for small data, adjust for large datasets
epochs = global_steps // steps_per_epoch

model.fit(dataset, truth, batch_size=batch_size, epochs=epochs)

model.save('/mnt/Velocity_Vault/Autofocus/Model/Tes_Model.keras')  # Save in SavedModel format


from tensorflow.keras.models import load_model

loaded_model = load_model('/mnt/Velocity_Vault/Autofocus/Model/Tes_Model.keras',custom_objects={'ordinal_loss': ordinal_loss})  # Load a SavedModel

# Demo Test
test_dataset = dataset[:50]  # Example test set
predictions = loaded_model.predict(test_dataset)
np.save('/mnt/Velocity_Vault/Autofocus/Model/'+"test_pred.npy",predictions)
# Display predictions
print("Test Predictions (logits):", predictions)


In [ ]:
# print(truth.shape)
# import numpy as np
# predictions=np.load('/mnt/Velocity_Vault/Autofocus/Model/'+"test_pred.npy")

print(type(predictions))
from pprint import pprint
pprint(predictions)

In [ ]:
pred=predictions[0]

print(pred)

In [ ]:
import numpy as np
from pprint import pprint

def calculate_p(x_list):
    return 1 / (1 + np.exp(-x_list))

prob=calculate_p(predictions)

#print("x values:", pred)
pprint((prob.shape))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_horizontal_bar(data, labels=None, title="Horizontal Bar Graph"):
    
    data=[d+1 for d in data]

    if labels is None:
        labels = [f"Frame {i+1}" for i in range(len(data))]
    
    y_positions = np.arange(len(data))
    
    plt.figure(figsize=(8, len(data) * 0.5))  # Adjust figure size based on the number of items
    plt.barh(y_positions, data, color="skyblue", edgecolor="black")
    
    plt.yticks(y_positions, labels)
    plt.xlim(0, max(data) + 1)  # Start x-axis at -1 and end a bit beyond the largest data value
    plt.axvline(0, color="black", linewidth=0.8)  # Add a vertical line at x=0 for reference
    plt.xlabel("Value")
    plt.ylabel("Items")
    plt.title(title)
    plt.tight_layout()
    plt.show()

# Example Usage
data = [0.2, 0.5, -0.3, 1.0, -0.8]
labels = ["A", "B", "C", "D", "E"]
plot_horizontal_bar(pred, title="Pred Graph")
plot_horizontal_bar(prob, title="Prob Graph")